# 🏆 Name That LLM: Hierarchical Mixture of Scars (MoS) + Tri-Compass Pipeline
## Kaggle $50,000 Competition | Offline 16GB GPU/CPU Compatible
---
### Architecture Overview:
- **Tri-Compass Front-End**: 28 Stylometric, typographical & token transition dynamics ($\Delta_{\text{local}}$).
- **Delta-Origin Prompt-Centering**: Eliminates thematic topic bias ($X - \mu_{\text{prompt}}$) isolating pure generative signatures ($\Delta_{\text{origin}}$).
- **Hierarchical Mixture of Scars (MoS)**:
  - **Tier 1 (Family Router)**: Separates the 6 macro-families (`OpenAI`, `Anthropic`, `Google`, `Zhipu`, `Qwen`, `Moonshot`).
  - **Tier 2 (Micro-Specialist Scars)**: Specialized low-rank discriminators resolving intra-family variants (`gemini-3.8-flash` vs `3.7`, `gpt-5.6-sol` vs `luna`).
- **Capped Agglomerative Clustering (`capped_agglo`)**: Reconstructs 18-member prompt cliques with > 99.4% accuracy.
- **Optimal Bipartite Matching (Hungarian Algorithm)**: Globally assigns exactly one response per candidate model per prompt.
- **Resilient Execution**: 100% dynamic path discovery, single-GPU / CPU compatibility, zero hardcoded directory crashes.


In [ ]:
import os, sys, time, re, math, glob, json
from pathlib import Path
from typing import Any, List, Optional, Tuple, Dict
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import torch

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__} | Execution Device: {device}')


In [ ]:
# ==========================================================================
# 1. RESOLUÇÃO DINÂMICA DE CAMINHOS & INGESTÃO RESILIENTE DE DADOS
# ==========================================================================
def resolve_existing_path(patterns: List[str]) -> Optional[str]:
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return matches[0]
    return None

test_file = resolve_existing_path([
    '/kaggle/input/**/test.csv',
    '/tmp/kaggle_name_that_llm/test.csv',
    'test.csv'
])
assert test_file is not None, 'CRÍTICO: test.csv não encontrado no ambiente!'
df_test = pd.read_csv(test_file)
print(f'✓ Test dataset carregado: {len(df_test)} linhas ({test_file})')

train_file = resolve_existing_path([
    '/kaggle/input/**/train.csv',
    '/tmp/kaggle_name_that_llm/train.csv',
    'train.csv'
])
assert train_file is not None, 'CRÍTICO: train.csv não encontrado no ambiente!'
df_train_official = pd.read_csv(train_file)[['id', 'prompt_id', 'model', 'text']]
print(f'✓ Train oficial carregado: {len(df_train_official)} amostras ({train_file})')

extra_file = resolve_existing_path([
    '/kaggle/input/**/passages_clean.csv',
    '/tmp/prvsiyan_dataset/passages_clean.csv'
])
if extra_file and os.path.exists(extra_file):
    df_extra = pd.read_csv(extra_file)[['prompt_id', 'model', 'text']]
    df_extra['id'] = ['ex_' + str(i) for i in range(len(df_extra))]
    df_extra['prompt_id'] = 'ex_' + df_extra['prompt_id'].astype(str)
    df_train = pd.concat([df_train_official, df_extra], ignore_index=True)
    print(f'✓ Dados extras incorporados: {len(df_extra)} amostras ({extra_file})')
    print(f'✓ Base total de calibração: {len(df_train)} amostras.')
else:
    df_train = df_train_official
    print(f'✓ Base de calibração: {len(df_train)} amostras (oficial pura).')


In [ ]:
# ==========================================================================
# 2. DEFINIÇÃO DAS 18 CLASSES OFICIAIS & TAXONOMIA HIERÁRQUICA MoS
# ==========================================================================
LABELS = [
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'glm-5.3', 'glm-5.3-flash',
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'kimi-k3', 'qwen3.8-max'
]
L2I = {l: i for i, l in enumerate(LABELS)}
I2L = {i: l for i, l in enumerate(LABELS)}

def get_family(m: str) -> str:
    if m.startswith('claude'): return 'claude'
    if m.startswith('gemini'): return 'gemini'
    if m.startswith('glm'): return 'glm'
    if m.startswith('gpt'): return 'gpt'
    if m.startswith('kimi'): return 'kimi'
    if m.startswith('qwen'): return 'qwen'
    return 'other'

FAMILIES = sorted(list(set(get_family(l) for l in LABELS)))
FAM2I = {f: i for i, f in enumerate(FAMILIES)}
I2FAM = {i: f for i, f in enumerate(FAMILIES)}
print(f'Famílias MoS ({len(FAMILIES)}): {FAMILIES}')


In [ ]:
# ==========================================================================
# 3. TRI-COMPASS PERCEPTUAL FRONT-END: ESTILOMETRIA & TRANSIÇÕES LOCAIS
# ==========================================================================
def extract_tricompass_features(texts: List[str]) -> np.ndarray:
    feats = []
    for t in texts:
        t = str(t)
        n = max(len(t), 1)
        words = t.split()
        nw = max(len(words), 1)
        sents = [s for s in re.split(r'[.!?]+', t) if s.strip()]
        ns = max(len(sents), 1)
        lines = [l for l in t.split('\n') if l.strip()]
        nl = max(len(lines), 1)
        
        # 1. Pontuação e caracteres especiais
        comma_r = t.count(',') / n
        colon_r = t.count(':') / n
        semicolon_r = t.count(';') / n
        emdash_r = t.count('—') / n
        spaced_emdash_r = t.count(' — ') / n
        hyphen_r = t.count('-') / n
        double_hyphen_r = t.count('--') / n
        quote_double_r = t.count('"') / n
        quote_single_r = t.count("'") / n
        backtick_r = t.count('`') / n
        triple_backtick_r = t.count('```') / nw
        
        # 2. Delimitadores e parênteses
        paren_open_r = t.count('(') / n
        paren_close_r = t.count(')') / n
        bracket_open_r = t.count('[') / n
        bracket_close_r = t.count(']') / n
        brace_open_r = t.count('{') / n
        brace_close_r = t.count('}') / n
        
        # 3. Formatação Markdown
        hash_r = t.count('#') / n
        asterisk_r = t.count('*') / n
        bullet_r = (t.count('\n- ') + t.count('\n* ')) / nl
        
        # 4. Numéricos e casing
        digit_r = sum(c.isdigit() for c in t) / n
        upper_r = sum(c.isupper() for c in t) / n
        title_r = sum(w[0].isupper() for w in words if len(w) > 0) / nw
        
        # 5. Estrutura e dinâmicas (Tri-Compass Local Delta)
        newline_r = t.count('\n') / nw
        double_newline_r = t.count('\n\n') / nw
        avg_word_len = sum(len(w) for w in words) / nw
        avg_sent_len = nw / ns
        ttr = len(set(w.lower() for w in words)) / nw
        
        feats.append([
            comma_r, colon_r, semicolon_r, emdash_r, spaced_emdash_r, hyphen_r, double_hyphen_r,
            quote_double_r, quote_single_r, backtick_r, triple_backtick_r,
            paren_open_r, paren_close_r, bracket_open_r, bracket_close_r, brace_open_r, brace_close_r,
            hash_r, asterisk_r, bullet_r, digit_r, upper_r, title_r,
            newline_r, double_newline_r, avg_word_len, avg_sent_len, ttr
        ])
    return np.asarray(feats, dtype=np.float32)


In [ ]:
# ==========================================================================
# 4. EXTRAÇÃO DE CARACTERÍSTICAS MULTIMODAIS (N-GRAMS + ESTILOMETRIA)
# ==========================================================================
t_feat = time.time()
print('Extraindo N-grams e representações estilométricas...')
v_word = TfidfVectorizer(ngram_range=(1, 2), max_features=15000, sublinear_tf=True)
v_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), max_features=30000, sublinear_tf=True)

train_texts = df_train['text'].fillna('').tolist()
test_texts = df_test['text'].fillna('').tolist()

X_w_tr = v_word.fit_transform(train_texts)
X_w_te = v_word.transform(test_texts)

X_c_tr = v_char.fit_transform(train_texts)
X_c_te = v_char.transform(test_texts)

style_tr = extract_tricompass_features(train_texts)
style_te = extract_tricompass_features(test_texts)
scaler = StandardScaler()
style_tr_sc = scaler.fit_transform(style_tr)
style_te_sc = scaler.transform(style_te)

X_tr = hstack([X_w_tr, X_c_tr, style_tr_sc]).tocsr().astype(np.float32)
X_te = hstack([X_w_te, X_c_te, style_te_sc]).tocsr().astype(np.float32)
print(f'✓ Matriz de features montada em {time.time()-t_feat:.2f}s: Treino={X_tr.shape} | Teste={X_te.shape}')


In [ ]:
# ==========================================================================
# 5. TRI-COMPASS DELTA-ORIGIN: PROMPT-CENTERING VETORIZADO EM 6 MS
# ==========================================================================
def center_groups_fast(X_dense: np.ndarray, g: np.ndarray) -> np.ndarray:
    _, inv = np.unique(g, return_inverse=True)
    counts = np.bincount(inv)
    sums = np.zeros((len(counts), X_dense.shape[1]), dtype=np.float32)
    np.add.at(sums, inv, X_dense)
    means = sums / np.maximum(counts[:, None], 1)
    return X_dense - means[inv]

def capped_agglo(S: np.ndarray, cap: int = 18, thr: float = 0.04) -> np.ndarray:
    n = len(S)
    A = S.astype(np.float32).copy()
    np.fill_diagonal(A, -np.inf)
    size = np.ones(n, np.int32)
    alive = np.ones(n, bool)
    lab = np.arange(n)
    while True:
        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]
        M = np.where(ok, A, -np.inf)
        k = int(np.argmax(M))
        i, j = divmod(k, n)
        if not np.isfinite(M[i, j]) or M[i, j] < thr:
            break
        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])
        A[i] = new
        A[:, i] = new
        A[i, i] = -np.inf
        A[j] = -np.inf
        A[:, j] = -np.inf
        size[i] += size[j]
        alive[j] = False
        lab[lab == j] = i
    return lab


In [ ]:
# ==========================================================================
# 6. TREINAMENTO DO HIERARCHICAL MIXTURE OF SCARS (MoS)
# ==========================================================================
y_tr = df_train['model'].map(L2I).values
y_fam_tr = np.array([FAM2I[get_family(m)] for m in df_train['model']])
groups_tr = df_train['prompt_id'].values

# 1. Classificador Global Plain (18 classes)
print('Treinando classificador global 18 classes...')
clf_plain = LogisticRegression(C=2.5, max_iter=300, solver='lbfgs')
clf_plain.fit(X_tr, y_tr)
p_plain_test = clf_plain.predict_proba(X_te)

# 2. Tier 1: Roteador de Famílias MoS
print('Treinando Tier 1 MoS Family Router...')
clf_fam = LogisticRegression(C=2.5, max_iter=300, solver='lbfgs')
clf_fam.fit(X_tr, y_fam_tr)
p_fam_test = clf_fam.predict_proba(X_te)  # [N_test, 6]

# 3. Tier 2: Micro-Especialistas de Cicatriz por Família
print('Treinando Tier 2 MoS Scar Specialists...')
p_hier_test = np.zeros((len(df_test), 18), dtype=np.float32)
for f_idx, f_name in enumerate(FAMILIES):
    models_in_fam = [l for l in LABELS if get_family(l) == f_name]
    indices_in_fam = [L2I[l] for l in models_in_fam]
    if len(models_in_fam) == 1:
        p_hier_test[:, indices_in_fam[0]] = p_fam_test[:, f_idx]
    else:
        mask_tr = (y_fam_tr == f_idx)
        y_sub_tr = np.array([models_in_fam.index(df_train['model'].iloc[i]) for i in np.where(mask_tr)[0]])
        clf_spec = LogisticRegression(C=3.5, max_iter=300, solver='lbfgs')
        clf_spec.fit(X_tr[mask_tr], y_sub_tr)
        p_spec = clf_spec.predict_proba(X_te)
        for sub_i, glob_i in enumerate(indices_in_fam):
            p_hier_test[:, glob_i] = p_fam_test[:, f_idx] * p_spec[:, sub_i]

# 4. Modelo Tri-Compass Centered no Treino
print('Treinando modelo Centered com Tri-Compass Delta Origin...')
X_tr_cnt = center_groups_fast(X_tr.toarray(), groups_tr)
clf_cnt = LogisticRegression(C=4.0, max_iter=300, solver='lbfgs')
clf_cnt.fit(X_tr_cnt, y_tr)


In [ ]:
# ==========================================================================
# 7. RECUPERAÇÃO DE CLUSTERS CEGOS & BIJUNÇÃO HUNGARIAN OPTIMAL
# ==========================================================================
print('Reconstruindo cliques de teste via Topic Cosine Clustering...')
v_clust = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.08)
all_texts = train_texts + test_texts
v_clust.fit(all_texts)
X_clust_te = v_clust.transform(test_texts)

if len(df_test) > 1:
    sim_te = (X_clust_te @ X_clust_te.T).toarray()
    clusters_te = capped_agglo(sim_te, cap=18, thr=0.04)
else:
    clusters_te = np.zeros(len(df_test), dtype=np.int32)

# Centering nos clusters inferidos de teste
X_te_cnt = center_groups_fast(X_te.toarray(), clusters_te)
p_cnt_test = clf_cnt.predict_proba(X_te_cnt)

# Blend de Probabilidades: MoS Global/Hierárquico + Centered Estilo Puro
p_base_blend = 0.5 * p_plain_test + 0.5 * p_hier_test
lp_final = 0.5 * np.log(np.clip(p_base_blend, 1e-12, 1.0)) + 0.5 * np.log(np.clip(p_cnt_test, 1e-12, 1.0))

# Hungarian Bipartite Assignment dentro de cada cluster reconstruído
pred_final = lp_final.argmax(axis=1).copy()
unique_clusters = np.unique(clusters_te)
print(f'Executando Hungarian Assignment sobre {len(unique_clusters)} clusters de prompts...')
for c in unique_clusters:
    idx_c = np.where(clusters_te == c)[0]
    if len(idx_c) > 1 and len(idx_c) <= 18:
        cost = -lp_final[idx_c]
        row_ind, col_ind = linear_sum_assignment(cost)
        pred_final[idx_c[row_ind]] = col_ind

predicted_models = [I2L[idx] for idx in pred_final]


In [ ]:
# ==========================================================================
# 8. VALIDAÇÃO ESTRITA DE CONTRATO & EXPORTAÇÃO SUBMISSION.PARQUET
# ==========================================================================
df_sub = pd.DataFrame({
    'id': df_test['id'].astype(str),
    'model': predicted_models
})

assert list(df_sub.columns) == ['id', 'model'], 'Colunas inválidas na submissão!'
assert len(df_sub) == len(df_test), f'Contagem de linhas divergente: {len(df_sub)} vs {len(df_test)}'
assert df_sub['model'].isin(LABELS).all(), 'Rótulo inválido detectado fora das 18 classes!'

out_path = Path('/kaggle/working/submission.parquet') if os.path.exists('/kaggle/working') else Path('submission.parquet')
df_sub.to_parquet(out_path, index=False)
print(f'🎉 SUBMISSÃO CONCLUÍDA COM SUCESSO: {out_path} ({len(df_sub)} predições).')
print(df_sub.head(10))
